# 🫁 NIH Chest X-Ray Classification — Fast Single-Session Run

| Steg | Tid |
|---|---|
| Setup + Kaggle auth | 2 min |
| Last ned 42 GB zip | 25-35 min |
| Pakk ut + resize til 224×224 | 15-20 min |
| Slett zip → frigjør 42 GB | 1 min |
| Tren 3 epoker (ResNet50, batch 128, FP16) | 45-60 min |
| Evaluering + lagring | 5 min |

**Totalt: ~100-120 min**

## Optimaliseringer for hastighet

1. **Resize til endelig 224×224 én gang** — ikke per batch under trening. Sparer ~10 min per epoke.
2. **Batch size 128 + FP16** — utnytter T4 GPU maksimalt (~10 GB minne brukt).
3. **3 epoker, ikke 5** — ResNet50 + transfer learning konvergerer raskt; epoke 4-5 gir bare 0.005-0.01 AUC ekstra.
4. **`compress_level=1`** ved PNG-skriving — 3× raskere enn default uten merkbart større filer.
5. **`Image.BILINEAR`** ved resize — 2× raskere enn LANCZOS, irrelevant kvalitetsforskjell på 1024→224.
6. **Slett zip umiddelbart** etter utpakking — gir trygg margin på disk.

## Hvis GPU-en forsvinner midt i

Modellen lagres til Drive etter hver epoke. Du har minst delvise resultater.

---


# 1. Setup

In [ ]:
import torch, os, time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU-minne: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("Ingen GPU! Runtime → Change runtime type → T4 GPU")

# Sjekk diskplass
import shutil
total, used, free = shutil.disk_usage('/content')
print(f"\nDisk: {used/1e9:.1f} GB brukt, {free/1e9:.1f} GB ledig")
assert free / 1e9 > 50, "Trenger minst 50 GB ledig disk for 42 GB zip + utpakking"


In [ ]:
# Drive for å lagre modell + resultater
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/nih_chest_xray_project'
DATA_DIR = '/content/data'
IMAGES_DIR = f'{DATA_DIR}/images'
CHECKPOINT_PATH = f'{DRIVE_DIR}/best_model.pt'

for d in [DATA_DIR, IMAGES_DIR, DRIVE_DIR, f'{DRIVE_DIR}/results']:
    os.makedirs(d, exist_ok=True)
print("✅ Mapper klare")


In [ ]:
!pip install -q kaggle gradio
print("✅ Pakker installert")


# 2. Kaggle-autentisering

Bruker Colab Secrets (🔑-ikon i venstre sidebar).
Trenger `KAGGLE_USERNAME` og `KAGGLE_KEY` fra `kaggle.json`.

In [ ]:
from google.colab import userdata
import json, pathlib

kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / 'kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY'),
}))
(kaggle_dir / 'kaggle.json').chmod(0o600)

# Rask test
!kaggle datasets list -s "nih chest" --max-size 1 > /dev/null && echo "✅ Kaggle OK"


# 3. Last ned metadata (få MB)

In [ ]:
import subprocess, zipfile

for fname in ['Data_Entry_2017.csv', 'train_val_list.txt', 'test_list.txt']:
    if os.path.exists(f'{DATA_DIR}/{fname}'):
        continue
    subprocess.run(['kaggle', 'datasets', 'download', 'nih-chest-xrays/data',
                    '-f', fname, '-p', DATA_DIR, '--force', '--quiet'], check=True)
    zip_path = f'{DATA_DIR}/{fname}.zip'
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA_DIR)
        os.remove(zip_path)
print("✅ Metadata lastet ned")


# 4. Last labels og bygg train/val/test-splits

In [ ]:
import pandas as pd
import numpy as np

DISEASE_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass',
    'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
NUM_CLASSES = len(DISEASE_LABELS)

df = pd.read_csv(f'{DATA_DIR}/Data_Entry_2017.csv')
for d in DISEASE_LABELS:
    df[d] = df['Finding Labels'].apply(lambda x: int(d in x))

# Pasient-respektert splitting (ingen leakage)
np.random.seed(42)
with open(f'{DATA_DIR}/train_val_list.txt') as f:
    tv_files = set(l.strip() for l in f if l.strip())
with open(f'{DATA_DIR}/test_list.txt') as f:
    test_files = set(l.strip() for l in f if l.strip())

tv_df = df[df['Image Index'].isin(tv_files)].copy()
test_df = df[df['Image Index'].isin(test_files)].copy()

patients = tv_df['Patient ID'].unique()
np.random.shuffle(patients)
val_size = int(0.1 * len(patients))
val_patients = set(patients[:val_size])

train_df = tv_df[~tv_df['Patient ID'].isin(val_patients)].copy()
val_df   = tv_df[ tv_df['Patient ID'].isin(val_patients)].copy()

print(f"Train: {len(train_df):>7,}  ({train_df['Patient ID'].nunique():,} pasienter)")
print(f"Val  : {len(val_df):>7,}  ({val_df['Patient ID'].nunique():,} pasienter)")
print(f"Test : {len(test_df):>7,}  ({test_df['Patient ID'].nunique():,} pasienter)")


# 5. Last ned zip (~30 min)

Hele datasettet kommer som én zip på 42 GB. Last ned først, pakk ut etterpå.

In [ ]:
ZIP_PATH = f'{DATA_DIR}/data.zip'

if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / 1e9
    print(f"Zip finnes allerede ({size_gb:.1f} GB), hopper over nedlasting")
else:
    print("Laster ned 42 GB zip — tar 25-35 min...")
    t0 = time.time()
    !kaggle datasets download nih-chest-xrays/data -p {DATA_DIR} --force
    elapsed = time.time() - t0
    print(f"\n✅ Nedlastet på {elapsed/60:.1f} min")

total, used, free = shutil.disk_usage('/content')
print(f"Disk: {used/1e9:.1f} GB brukt, {free/1e9:.1f} GB ledig")


# 6. Pakk ut + resize til 224×224 (~15-20 min)

**Tre kritiske optimaliseringer:**

1. **Resize til endelig 224×224** her — ikke under trening. Sparer ~10 min per epoke.
2. **`Image.BILINEAR`** — 2× raskere enn LANCZOS, ingen praktisk kvalitetsforskjell.
3. **`compress_level=1`** — 3× raskere PNG-skriving uten merkbart større filer.

Etter ferdig: ~5 GB med bilder. Zipen slettes umiddelbart i neste celle.

In [ ]:
import io, glob
from PIL import Image
from tqdm.auto import tqdm

IMG_SIZE = 224  # ResNet50 input — gjør resize én gang nå, ikke per batch

t0 = time.time()
processed = skipped = errors = 0

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    image_names = [n for n in zf.namelist() if n.lower().endswith('.png')]
    print(f"Fant {len(image_names):,} bilder i zip")

    pbar = tqdm(image_names, desc='Resize', smoothing=0.1)
    for name in pbar:
        out_path = f'{IMAGES_DIR}/{os.path.basename(name)}'
        if os.path.exists(out_path):
            skipped += 1
            continue
        try:
            with zf.open(name) as f:
                img = Image.open(io.BytesIO(f.read()))
                img = img.convert('L').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                img.save(out_path, 'PNG', compress_level=1)
            processed += 1
        except Exception as e:
            errors += 1
            if errors < 3:
                print(f"\n  Feil på {os.path.basename(name)}: {e}")

elapsed = time.time() - t0
print(f"\n✅ Ferdig på {elapsed/60:.1f} min")
print(f"   Prosessert: {processed:,}  Skipped: {skipped:,}  Feil: {errors}")
print(f"   Bilder i mappe: {len(os.listdir(IMAGES_DIR)):,}")


# 7. Slett zip → frigjør 42 GB

In [ ]:
if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / 1e9
    os.remove(ZIP_PATH)
    print(f"🗑️  Slettet zip ({size_gb:.1f} GB frigjort)")

total, used, free = shutil.disk_usage('/content')
print(f"Disk nå: {used/1e9:.1f} GB brukt, {free/1e9:.1f} GB ledig")

# Bygg filename → path-oppslag
all_pngs = glob.glob(f'{IMAGES_DIR}/*.png')
filename_to_path = {os.path.basename(p): p for p in all_pngs}
print(f"\nBilder klar for trening: {len(filename_to_path):,}")
assert len(filename_to_path) >= 110000, "For få bilder!"


# 8. Dataset + DataLoaders

**Viktig:** Siden bildene allerede er 224×224, hopper vi over Resize i transformene.
Det sparer ~30% datalastingstid per batch.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class ChestXrayDataset(Dataset):
    def __init__(self, df, filename_to_path, transform):
        self.df = df.reset_index(drop=True)
        self.f2p = filename_to_path
        self.transform = transform
        self.labels = self.df[DISEASE_LABELS].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.f2p[self.df.iloc[idx]['Image Index']]
        # convert('RGB') dupliserer gråtone til 3 kanaler (ResNet forventer 3)
        img = Image.open(path).convert('RGB')
        return self.transform(img), torch.from_numpy(self.labels[idx])

# Trening: bare augmentasjon, ingen resize (bildene er allerede 224×224)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = ChestXrayDataset(train_df, filename_to_path, train_transform)
val_ds   = ChestXrayDataset(val_df,   filename_to_path, eval_transform)
test_ds  = ChestXrayDataset(test_df,  filename_to_path, eval_transform)

# Test at det funker
img, lbl = train_ds[0]
print(f"Sample: {img.shape}, label sum: {lbl.sum().item()}")


In [ ]:
BATCH_SIZE = 128  # T4 har 16 GB, FP16 + ResNet50 ved batch 128 ≈ 10 GB
NUM_WORKERS = 4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

print(f"Batches per epoch — train: {len(train_loader):,} | val: {len(val_loader):,}")


# 9. Modell, loss, optimizer

**Multi-label setup:**
- `BCEWithLogitsLoss` (én sigmoid per klasse, ikke softmax)
- `pos_weight` kompenserer for at f.eks. Hernia bare har 0.2% positive eksempler
- AUC per sykdom som metrikk (accuracy er meningsløst med 54% "No Finding")

In [ ]:
from torchvision import models

device = torch.device('cuda')

# Pretrained ResNet50, ny final-layer
weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)
model.fc = torch.nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

# pos_weight: kompenserer for klasseubalanse
pos_counts = train_df[DISEASE_LABELS].sum().values
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float32, device=device)

NUM_EPOCHS = 3   # Transfer learning konvergerer raskt; epoke 4-5 gir minimalt ekstra
LR = 1e-4

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * len(train_loader))
scaler    = torch.amp.GradScaler('cuda')

print(f"Modell klar | Parametre: {sum(p.numel() for p in model.parameters()):,}")
print(f"Epoker: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}")


# 10. Treningsløkke

**~15-20 min per epoke** med batch 128 + FP16 på T4.

In [ ]:
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

def train_epoch():
    model.train()
    total_loss, n = 0.0, 0
    pbar = tqdm(train_loader, desc='Train', leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        pbar.set_postfix(loss=f'{total_loss/n:.4f}')
    return total_loss / n

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits, all_labels = [], []
    for x, y in tqdm(loader, desc='Eval', leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_logits.append(logits.float().cpu())
        all_labels.append(y.cpu())
    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    probs_np = 1 / (1 + np.exp(-logits_np))
    aucs = []
    for i in range(NUM_CLASSES):
        if 0 < labels_np[:, i].sum() < len(labels_np):
            aucs.append(roc_auc_score(labels_np[:, i], probs_np[:, i]))
        else:
            aucs.append(float('nan'))
    return total_loss / n, float(np.nanmean(aucs)), aucs, probs_np, labels_np

print("✅ Trenings-/eval-funksjoner klare")


In [ ]:
best_val_auc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

print(f"Starter trening — forventet ~{NUM_EPOCHS * 18} min total\n")
overall_t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    print(f"=== Epoke {epoch}/{NUM_EPOCHS} ===")

    train_loss = train_epoch()
    val_loss, val_auc, val_aucs, _, _ = evaluate(val_loader)

    elapsed = time.time() - t0
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    print(f"  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_AUC={val_auc:.4f}  ({elapsed/60:.1f} min)")

    # Lagre etter HVER epoke (sikkerhetsnett mot GPU-timeout)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'val_auc': val_auc,
        'val_aucs': val_aucs,
        'history': history,
        'disease_labels': DISEASE_LABELS,
        'is_best': val_auc > best_val_auc,
    }, CHECKPOINT_PATH)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        print(f"  💾 Ny beste modell (val_AUC={val_auc:.4f})")
    else:
        print(f"  💾 Lagret (ikke bedre enn {best_val_auc:.4f})")
    print()

print(f"🎉 Trening ferdig på {(time.time()-overall_t0)/60:.1f} min")
print(f"   Beste val-AUC: {best_val_auc:.4f}")


# 11. Evaluering på testsettet

In [ ]:
# Last beste modell
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Lastet modell fra epoke {ckpt['epoch']} (val_AUC={ckpt['val_auc']:.4f})")

test_loss, test_auc, test_aucs, test_probs, test_labels = evaluate(test_loader)

print(f"\n📊 Test-resultater:")
print(f"   Mean AUC: {test_auc:.4f}\n")

auc_df = pd.DataFrame({
    'Sykdom': DISEASE_LABELS,
    'AUC': [round(a, 4) for a in test_aucs],
    'Positive': test_labels.sum(axis=0).astype(int),
}).sort_values('AUC', ascending=False)
print(auc_df.to_string(index=False))

auc_df.to_csv(f'{DRIVE_DIR}/results/test_results.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# AUC per sykdom
auc_sorted = auc_df.sort_values('AUC')
colors = plt.cm.RdYlGn(np.array(auc_sorted['AUC'].values, dtype=float))
axes[0].barh(auc_sorted['Sykdom'], auc_sorted['AUC'], color=colors)
axes[0].axvline(0.5, color='gray', ls='--', label='Random')
axes[0].axvline(test_auc, color='navy', ls='--', label=f'Mean ({test_auc:.3f})')
axes[0].set_xlabel('AUC')
axes[0].set_title('Test AUC per sykdom')
axes[0].legend()

# ROC-kurver for utvalgte sykdommer
for d in ['Cardiomegaly', 'Edema', 'Pneumonia', 'Pneumothorax', 'Effusion', 'Hernia']:
    i = DISEASE_LABELS.index(d)
    fpr, tpr, _ = roc_curve(test_labels[:, i], test_probs[:, i])
    axes[1].plot(fpr, tpr, label=f'{d} ({test_aucs[i]:.3f})')
axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC-kurver'); axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/results/roc_curves.png', dpi=120)
plt.show()

# Trenings-historikk
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history['train_loss'], 'o-', label='Train')
ax[0].plot(history['val_loss'],   'o-', label='Val')
ax[0].set(title='Loss', xlabel='Epoke'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history['val_auc'], 'o-g')
ax[1].set(title='Val AUC', xlabel='Epoke'); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/results/training_history.png', dpi=120)
plt.show()

pd.DataFrame(history).to_csv(f'{DRIVE_DIR}/results/training_history.csv', index=False)
print(f"\n💾 Alle resultater lagret til {DRIVE_DIR}/results/")


# 12. Gradio-demo (valgfri)

In [ ]:
# ============================================================
# AVANSERT GRADIO-DEMO MED GRADCAM + KLINISKE FORKLARINGER
# ============================================================

import gradio as gr
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import io

# ── 1. KLINISKE FORKLARINGER PER SYKDOM ────────────────────────────────────
DISEASE_INFO = {
    "Atelectasis": {
        "no": "Atelektase",
        "what": "Delvis eller fullstendig kollaps av lungevev. Luftblærene (alveoler) folder seg sammen.",
        "looks": "Økt tetthet (hvitere område) i en del av lungen, ofte med forskyvning av strukturer.",
        "where": "Vanligst i nedre lungelapper, særlig bak og til høyre.",
    },
    "Cardiomegaly": {
        "no": "Kardiomegali",
        "what": "Forstørret hjerte. Kan skyldes hjertesvikt, høyt blodtrykk eller klaffefeil.",
        "looks": "Hjertets bredde er mer enn 50% av brystkassens indre bredde (cardiothoracic ratio > 0.5).",
        "where": "Midten av brystet — hjertet tar mer plass enn normalt.",
    },
    "Consolidation": {
        "no": "Konsolidering",
        "what": "Luften i lungevevet erstattes av væske, puss eller blod. Vanlig ved bakteriell pneumoni.",
        "looks": "Tett, hvitt område uten synlig luftstruktur inni. Kan ha 'air bronchogram'.",
        "where": "Kan ramme ett segment, én lapp eller hele lungen.",
    },
    "Edema": {
        "no": "Lungeødem",
        "what": "Væske lekker fra blodårene inn i lungevevet. Oftest tegn på hjertesvikt.",
        "looks": "Jevnt uklar, tåkete lunge særlig rundt lungeroten. Kalles 'butterfly pattern'.",
        "where": "Symmetrisk, starter sentralt og sprer seg utover.",
    },
    "Effusion": {
        "no": "Pleural effusjon",
        "what": "Væske samler seg i pleurahulen — rommet mellom lungen og brystkasseveggen.",
        "looks": "Grå/hvit homogen fortetting i nedre del av brystet med jevn øvre grense.",
        "where": "Nedre deler av brystet, oftest langs sidene.",
    },
    "Emphysema": {
        "no": "Emfysem",
        "what": "Destruksjon av luftblærene — lungen blir overblåst. Vanligste årsak: røyking.",
        "looks": "Mørke overblåste lunger, flat mellomgulv, få kar synlige.",
        "where": "Øvre lungelapper rammes oftest.",
    },
    "Fibrosis": {
        "no": "Lungefibrose",
        "what": "Arrvev erstatter normalt lungevev. Lungen blir stiv og mister elastisitet.",
        "looks": "Nettverksliknende uregelmessige linjer (retikulært mønster). Volum redusert.",
        "where": "Oftest nedre og ytre deler av lungene.",
    },
    "Hernia": {
        "no": "Hernie",
        "what": "Mageorganer trenger opp gjennom mellomgulvet og inn i brystet.",
        "looks": "Uvanlige strukturer i brystet — kan ligne tarmsløyfer over mellomgulvet.",
        "where": "Oftest venstre side eller midtlinjen.",
    },
    "Infiltration": {
        "no": "Infiltrat",
        "what": "Ikke-spesifikk betegnelse: betennelse, væske eller blod fyller deler av lungevevet.",
        "looks": "Ulne, sky-liknende fortettinger. Ikke like tett som konsolidering.",
        "where": "Varierer — kan ramme enhver del av lungen.",
    },
    "Mass": {
        "no": "Masse",
        "what": "En avgrenset opakitet > 3 cm. Krever alltid videre utredning for malignitet.",
        "looks": "Godt avgrenset rundt eller lobulert område. Uregelmessige kanter ved kreft.",
        "where": "Kan sitte hvor som helst i lungen.",
    },
    "Nodule": {
        "no": "Nodule",
        "what": "En avgrenset opakitet ≤ 3 cm. Kan være godartet (granulom) eller ondartet.",
        "looks": "Lite, rundt, velavgrenset hvitt punkt. Kalk i kanten tyder på godarthet.",
        "where": "Kan sitte hvor som helst.",
    },
    "Pleural_Thickening": {
        "no": "Pleural fortykning",
        "what": "Lungehinnene er fortykket — ofte etter betennelse eller asbesteksponering.",
        "looks": "Hvit linje langs brystkasseveggen, jevnere enn effusjon.",
        "where": "Langs kanten av lungen opp mot brystkassevegg.",
    },
    "Pneumonia": {
        "no": "Lungebetennelse",
        "what": "Infeksjon i lungevevet. Kan skyldes bakterier, virus eller sopp.",
        "looks": "Konsolidering eller infiltrater — hvite skyer, ofte med air bronchogram.",
        "where": "Nedre lapper oftest. Lobar: én hel lapp. Bronkopneumoni: flekkvis.",
    },
    "Pneumothorax": {
        "no": "Pneumothorax",
        "what": "Luft lekker inn i pleurahulen. Lungen kollapser som en punktert ballong.",
        "looks": "Skarp linje (visceral pleura) parallelt med brystkassevegg. Ingen lungekar utenfor linjen.",
        "where": "Øvre del av lungen. Se etter linjen øverst i brystet.",
    },
}

CONFIDENCE_LEVELS = [
    (0.70, "🔴 Sterk indikasjon",   "Modellen er svært sikker på dette funnet."),
    (0.50, "🟠 Moderat indikasjon", "Tydelige tegn til stede — bør undersøkes nærmere."),
    (0.35, "🟡 Svak indikasjon",    "Noe usikkerhet. Kan være et tidlig eller subtilt funn."),
    (0.00, "⚪ Ikke påvist",         "Modellen ser ingen tydelige tegn til dette."),
]

def get_confidence(prob):
    for threshold, label, desc in CONFIDENCE_LEVELS:
        if prob >= threshold:
            return label, desc
    return CONFIDENCE_LEVELS[-1][1], CONFIDENCE_LEVELS[-1][2]


# ── 2. GRADCAM ──────────────────────────────────────────────────────────────
class GradCAM:
    """
    Viser HVILKE piksler i bildet som var viktigst for prediksjonen.
    Bruker gradienter fra det siste conv-laget i ResNet50 (layer4[-1]).
    Rødt = høy aktivasjon = modellen ser noe her.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, input_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

gradcam = GradCAM(model, model.layer4[-1])

def overlay_heatmap(pil_img, heatmap, alpha=0.45):
    img_np = np.array(pil_img.convert("RGB"))
    heatmap_rgb = (cm.get_cmap("jet")(heatmap)[:, :, :3] * 255).astype(np.uint8)
    return Image.fromarray((img_np * (1-alpha) + heatmap_rgb * alpha).astype(np.uint8))


# ── 3. PREDICT + EXPLAIN ────────────────────────────────────────────────────
def predict_and_explain(image):
    if image is None:
        return None, "Last opp et røntgenbilde."

    model.eval()
    img_resized = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    tensor = eval_transform(img_resized).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast("cuda"):
            logits = model(tensor)
        probs = torch.sigmoid(logits).float().cpu().numpy()[0]

    sorted_idx   = np.argsort(probs)[::-1]
    positive_idx = [i for i in sorted_idx if probs[i] >= 0.35]

    # GradCAM for topp-funn
    heatmap_img = None
    if len(positive_idx) > 0:
        t = eval_transform(img_resized).unsqueeze(0).to(device)
        cam = gradcam.generate(t, positive_idx[0])
        heatmap_img = overlay_heatmap(img_resized, cam)

    # ── Figur ──────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 10), facecolor="#1a1a2e")

    ax_orig = fig.add_axes([0.02, 0.35, 0.22, 0.55])
    ax_heat = fig.add_axes([0.02, 0.05, 0.22, 0.28])

    ax_orig.imshow(img_resized, cmap="gray")
    ax_orig.set_title("Røntgenbilde", color="white", fontsize=11, pad=6)
    ax_orig.axis("off")

    if heatmap_img:
        top_name = DISEASE_INFO[DISEASE_LABELS[positive_idx[0]]]["no"]
        ax_heat.imshow(heatmap_img)
        ax_heat.set_title(f"GradCAM: {top_name}\n(rødt = høy aktivasjon)", color="#ff9f43", fontsize=9, pad=4)
    else:
        ax_heat.text(0.5, 0.5, "Ingen funn\npåvist", ha="center", va="center", color="gray", fontsize=12)
        ax_heat.set_facecolor("#0f0f23")
    ax_heat.axis("off")

    # Sannsynlighetsbars
    ax_bar = fig.add_axes([0.28, 0.42, 0.38, 0.52])
    colors = []
    for i in sorted_idx[:8]:
        p = probs[i]
        if   p >= 0.70: colors.append("#e74c3c")
        elif p >= 0.50: colors.append("#e67e22")
        elif p >= 0.35: colors.append("#f1c40f")
        else:           colors.append("#2c3e50")

    bar_labels = [DISEASE_INFO[DISEASE_LABELS[i]]["no"] for i in sorted_idx[:8]]
    bar_vals   = [probs[i] for i in sorted_idx[:8]]
    bars = ax_bar.barh(bar_labels[::-1], bar_vals[::-1], color=colors[::-1], height=0.6)
    ax_bar.set_xlim(0, 1)
    ax_bar.axvline(0.50, color="white", lw=1, ls="--", alpha=0.4)
    ax_bar.axvline(0.35, color="yellow", lw=1, ls=":", alpha=0.4)
    for bar, val in zip(bars, bar_vals[::-1]):
        ax_bar.text(min(val+0.02, 0.97), bar.get_y()+bar.get_height()/2,
                    f"{val:.0%}", va="center", color="white", fontsize=10, fontweight="bold")
    ax_bar.set_facecolor("#0f0f23")
    ax_bar.tick_params(colors="white", labelsize=10)
    ax_bar.spines[:].set_color("#333355")
    ax_bar.set_title("Sannsynlighet per funn (topp 8)", color="white", fontsize=11, pad=8)

    # Forklaringsboks
    ax_text = fig.add_axes([0.68, 0.02, 0.30, 0.94])
    ax_text.set_facecolor("#0f0f23")
    ax_text.axis("off")

    if len(positive_idx) == 0:
        title_txt  = "✅ INGEN PATOLOGI PÅVIST"
        title_col  = "#2ecc71"
        body_txt   = (
            "Modellen finner ingen av de 14 sykdommene\n"
            "den er trent på å gjenkjenne.\n\n"
            "Dette kan bety:\n"
            " • Bildet ser normalt ut\n"
            " • Eventuelle funn er utenfor modellens\n"
            "   treningsdata (f.eks. brudd, TB)\n"
            " • Subtile funn under deteksjonsgrensen"
        )
    else:
        top_d  = DISEASE_LABELS[positive_idx[0]]
        info   = DISEASE_INFO[top_d]
        c_lbl, c_desc = get_confidence(probs[positive_idx[0]])
        title_txt = f"{c_lbl}\n{info['no'].upper()}"
        title_col = "#e74c3c" if probs[positive_idx[0]] >= 0.50 else "#f1c40f"
        body_txt  = (
            f"HVA ER DET?\n{info['what']}\n\n"
            f"HVORDAN SER DET UT?\n{info['looks']}\n\n"
            f"HVOR I BILDET?\n{info['where']}\n\n"
            f"MODELLENS SIKKERHET:\n{c_desc}"
        )
        if len(positive_idx) > 1:
            body_txt += "\n\nANDRE FUNN OVER 35%:"
            for i in positive_idx[1:4]:
                d = DISEASE_LABELS[i]
                body_txt += f"\n • {DISEASE_INFO[d]['no']}: {probs[i]:.0%}"

    ax_text.text(0.05, 0.97, title_txt, transform=ax_text.transAxes,
                 color=title_col, fontsize=12, fontweight="bold", va="top")
    ax_text.text(0.05, 0.75, body_txt, transform=ax_text.transAxes,
                 color="#ecf0f1", fontsize=9.5, va="top", linespacing=1.6)
    ax_text.text(0.05, 0.04, "⚠️  Kun for demo og læring.\nIkke for klinisk bruk.",
                 transform=ax_text.transAxes, color="#7f8c8d", fontsize=8, va="bottom")

    fig.text(0.50, 0.97, "🫁  NIH Chest X-Ray Analyse",
             ha="center", color="white", fontsize=14, fontweight="bold")
    fig.text(0.50, 0.93,
             f"ResNet50 | Transfer Learning | Test mean AUC: {test_auc:.3f} | 112 120 treningsbilder",
             ha="center", color="#7f8c8d", fontsize=9)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=130, bbox_inches="tight",
                facecolor="#1a1a2e", edgecolor="none")
    plt.close(fig)
    buf.seek(0)
    result_img = Image.open(buf).copy()
    buf.close()

    # Markdown-summary
    if len(positive_idx) == 0:
        md = "### ✅ Ingen patologi påvist\n\nModellen finner ingen av de 14 sykdommene i dette bildet."
    else:
        lines = ["### 🔍 Funn\n"]
        for i in positive_idx[:5]:
            d = DISEASE_LABELS[i]
            c_lbl, _ = get_confidence(probs[i])
            lines.append(f"**{DISEASE_INFO[d]['no']}** — {probs[i]:.0%} {c_lbl}")
            lines.append(f"> {DISEASE_INFO[d]['what']}\n")
        lines.append("---\n*⚠️ Kun for demo og læringsformål — ikke for klinisk bruk.*")
        md = "\n".join(lines)

    return result_img, md


# ── 4. GRADIO UI ────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Base(), title="NIH Chest X-Ray Analyse") as demo:
    gr.Markdown("""
    # 🫁 NIH Chest X-Ray Analyse
    **ResNet50 fine-tunet på 112 120 røntgenbilder | 14 thorax-sykdommer**

    Last opp et frontalt thorax-røntgenbilde for å få:
    - 📊 **Sannsynlighet** for alle 14 sykdommer
    - 🗺️ **GradCAM-heatmap** — viser *hvor* i bildet modellen ser patologi (rødt = høy aktivasjon)
    - 📖 **Klinisk forklaring** — hva funnet betyr, hvordan det ser ut, og hvor i bildet

    > ⚠️ Kun for læringsformål — ikke for klinisk diagnostikk.
    """)
    with gr.Row():
        inp = gr.Image(type="pil", label="Last opp røntgenbilde", height=320)
        btn = gr.Button("🔍 Analyser", variant="primary", scale=0)
    with gr.Row():
        out_img = gr.Image(label="Analyse med GradCAM", type="pil", height=520)
    out_md = gr.Markdown()

    btn.click(fn=predict_and_explain, inputs=inp, outputs=[out_img, out_md])
    inp.change(fn=predict_and_explain, inputs=inp, outputs=[out_img, out_md])

demo.launch(share=True)

In [ ]:
# Finn ett eksempelbilde per sykdom fra testsettet
import random
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image

# Velg sykdommer du vil se
showcase = ['Cardiomegaly', 'Effusion', 'Pneumothorax', 'Hernia', 'Emphysema']

fig, axes = plt.subplots(1, len(showcase), figsize=(18, 5))

for ax, disease in zip(axes, showcase):
    # Finn bilder i testsettet som KUN har denne sykdommen (renere test)
    subset = test_df[
        (test_df[disease] == 1) &
        (test_df[DISEASE_LABELS].sum(axis=1) == 1)  # bare én diagnose
    ]
    if len(subset) == 0:
        subset = test_df[test_df[disease] == 1]  # fallback: ta med multi-label også

    # Ta et tilfeldig bilde fra denne gruppen
    row = subset.sample(1).iloc[0]
    fname = row['Image Index']
    path = filename_to_path[fname]

    # Vis bilde
    img = Image.open(path).convert('RGB')
    ax.imshow(img, cmap='gray')
    ax.set_title(f"FASIT: {disease}", fontsize=10, color='green')
    ax.axis('off')

    # Kjør prediksjon
    tensor = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad(), torch.amp.autocast('cuda'):
        logits = model(tensor)
    probs = torch.sigmoid(logits).float().cpu().numpy()[0]

    # Skriv ut topp-3 prediksjoner
    top3_idx = probs.argsort()[::-1][:3]
    print(f"\n{disease} ({fname}):")
    print(f"  Fasit-sannsynlighet: {probs[DISEASE_LABELS.index(disease)]:.0%}")
    for i in top3_idx:
        marker = "✅" if DISEASE_LABELS[i] == disease else "  "
        print(f"  {marker} {DISEASE_LABELS[i]:25s}: {probs[i]:.0%}")

plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/results/showcase_predictions.png', dpi=120)
plt.show()